# V1 explainability — occlusion + LIME + leaf containment (all arms)

Runs `src/explain.py` on the trained checkpoints in `MyDrive/csg520_outputs/checkpoints`.
Results go to `MyDrive/csg520_outputs/explain/<arm>/` and the combined table to `explain/v1_table.csv`.

**Two runtimes, in parallel:**
- **Runtime A: GPU (T4 or L4).** Run sections 1–5 (A1, A3, A4). Allow about 10–20 min per arm.
- **Runtime B: CPU only** (Runtime → Change runtime type → CPU; no GPU units used). Run sections 1–4 and then 6 (A2 RF).
  Allow about 30–90 min, depending on the number of cores.

Progress is saved after every image. **After a disconnect, re-run all cells** and each arm resumes where it stopped.


## 1. Paths + Drive

In [ ]:
from pathlib import Path
CODE_DIR = Path("/content/drive/MyDrive/csg520_code/src")    # upload the updated src/ here (explain.py is new)
OUT_DIR  = Path("/content/drive/MyDrive/csg520_outputs")     # checkpoints/ and split/ live here
CKPT_DIR = OUT_DIR / "checkpoints"
LOCAL    = Path("/content/leaf_xai")

from google.colab import drive
drive.mount("/content/drive")
assert (CODE_DIR / "explain.py").exists(), f"upload the new explain.py into {CODE_DIR}"
for f in ("a1_resnet50.pt", "a3_cnn.pt", "a4_vit.pt", "a2_rf.joblib"):
    print(f"{f:<22}", "OK" if (CKPT_DIR / f).exists() else "MISSING")

## 2. Code → local disk, install lime/timm

In [ ]:
import shutil, os
shutil.copytree(CODE_DIR, LOCAL / "src", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns(".git", "__pycache__"))
!pip -q install lime "timm>=0.9" kagglehub
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (fine for A2)",
      "| CPU cores:", os.cpu_count())

## 3. PlantVillage colour images + the segmented variant (for the leaf masks)
The models were trained on the **colour** images only. The **segmented** images are used only to build the ground-truth leaf masks.

In [ ]:
import kagglehub
try:
    from google.colab import userdata
    os.environ.setdefault("KAGGLE_USERNAME", userdata.get("KAGGLE_USERNAME"))
    os.environ.setdefault("KAGGLE_KEY", userdata.get("KAGGLE_KEY"))
except Exception:
    pass
ds_path = Path(kagglehub.dataset_download("abdallahalidev/plantvillage-dataset"))

def find_variant(name):
    return next(p for p in sorted(ds_path.rglob(name))
                if p.is_dir() and sum(1 for c in p.iterdir() if c.is_dir()) == 38)

color, MASK_ROOT = find_variant("color"), find_variant("segmented")
raw = LOCAL / "data" / "raw"
raw.parent.mkdir(parents=True, exist_ok=True)
if not raw.exists():
    raw.symlink_to(color)
print("colour :", color)
print("masks  :", MASK_ROOT, "|", sum(1 for _ in MASK_ROOT.rglob("*_final_masked*")), "segmented files")

## 4. The frozen split (saved to Drive by the training notebook)

In [ ]:
import pandas as pd
for f in ("splits.csv", "classes.json"):
    shutil.copy(OUT_DIR / "split" / f, LOCAL / "data" / f)
counts = pd.read_csv(LOCAL / "data" / "splits.csv")["split"].value_counts().to_dict()
print(counts)
assert counts == {"train": 38013, "val": 8146, "test": 8146}

## 5. Runtime A (GPU): A1, A3, A4
Smoke test first (3 images, throwaway folder). Then the real runs: 380 images for occlusion and 114 for LIME per arm.
Each arm prints its V1 summary when it finishes.

In [ ]:
%cd /content/leaf_xai/src
!python explain.py --arms a3_cnn --limit 3 --ckpt-dir {CKPT_DIR} --output-dir /content/smoke --mask-root "{MASK_ROOT}" --fresh

In [ ]:
%cd /content/leaf_xai/src
!python explain.py --arms a1_resnet50 a3_cnn a4_vit --ckpt-dir {CKPT_DIR} --output-dir {OUT_DIR} --mask-root "{MASK_ROOT}" 2>&1 | tee -a {OUT_DIR}/explain_gpu_log.txt

*Optional extra arms, same command:* `a1_resnet50_scratch` (pretraining ablation, a direct test of A1's question) and `a1_efficientnet_b0`.
Each takes about the same time as the others.

In [ ]:
# %cd /content/leaf_xai/src
# !python explain.py --arms a1_resnet50_scratch a1_efficientnet_b0 --ckpt-dir {CKPT_DIR} --output-dir {OUT_DIR} --mask-root "{MASK_ROOT}"

## 6. Runtime B (CPU): A2 random forest
The RF checkpoint is about 1.2 GB, so loading takes a minute. For each image the features (including the ExG segmentation) are recomputed
on every perturbed copy, using all CPU cores. LIME runs on 2 images/class (76) for A2, as agreed.

In [ ]:
%cd /content/leaf_xai/src
!python explain.py --arms a2_rf --n-jobs -1 --ckpt-dir {CKPT_DIR} --output-dir {OUT_DIR} --mask-root "{MASK_ROOT}" 2>&1 | tee -a {OUT_DIR}/explain_a2_log.txt

## 7. The V1 table + example figures (run anywhere once the arms are done)

In [ ]:
%cd /content/leaf_xai/src
!python explain.py --summarize --output-dir {OUT_DIR}
import pandas as pd
pd.read_csv(OUT_DIR / "explain" / "v1_table.csv")

In [ ]:
from IPython.display import Image, display
for arm_dir in sorted((OUT_DIR / "explain").glob("a*")):
    if (arm_dir / "examples.png").exists():
        print(arm_dir.name); display(Image(str(arm_dir / "examples.png")))